# GRU — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## A compact gated recurrence blends the old state with a candidate state
The examples visualize update-gate interpolation, reset-gate forgetting, candidate construction, GRU versus vanilla decay, and a synthetic sequence decision.

In [ ]:
h_old=2.; h_tilde=-1.; z=np.array([0,0.25,0.5,0.75,1.0]); h=(1-z)*h_old+z*h_tilde
plt.figure(figsize=(6,3)); plt.plot(z,h,'-o'); plt.title('Update gate interpolates old and candidate')
assert h[0]==2 and h[-1]==-1

In [ ]:
h=3.; r=np.array([0,0.5,1.0]); seen=r*h
plt.figure(figsize=(5,3)); plt.bar(['r=0','r=.5','r=1'],seen); plt.title('Reset gate controls how much past reaches candidate')
assert np.allclose(seen,[0,1.5,3])

In [ ]:
x=1.; h=2.; r=.25; cand=np.tanh(0.7*x+0.4*r*h)
plt.figure(figsize=(4,3)); plt.bar(['candidate'],[cand]); plt.ylim(0,1); plt.title('Candidate uses reset-filtered state')
assert abs(cand-0.7162978701990245)<1e-9

In [ ]:
T=np.arange(1,31); vanilla=.5**T; gru=(1-.05)**T
plt.figure(figsize=(6,3)); plt.plot(T,vanilla,label='vanilla'); plt.plot(T,gru,label='GRU keep'); plt.legend(); plt.title('Small update gate preserves state')
assert gru[-1]>vanilla[-1]

In [ ]:
seq=np.array([1,0,0,0,1]); h=0.; hs=[]
for x in seq:
    z=0.2 if x==0 else 0.8; cand=np.tanh(x+0.5*h); h=(1-z)*h+z*cand; hs.append(h)
plt.figure(figsize=(6,3)); plt.plot(hs,'-o'); plt.title('GRU reacts strongly to informative tokens')
assert hs[-1]>hs[1]